In [ ]:
#출처 https://github.com/huggingface/sentence-transformers/blob/main/examples/sentence_transformer/applications/retrieve_rerank/retrieve_rerank_simple_wikipedia.ipynb

import gzip
import json
import os

import torch

from sentence_transformers import CrossEncoder, SentenceTransformer
from sentence_transformers.util import http_get, semantic_search

# 시맨틱 검색을 위해 Bi-Encoder 사용
bi_encoder = SentenceTransformer("sentence-transformers/multi-qa-MiniLM-L6-cos-v1")
bi_encoder.max_seq_length = 256  
top_k = 32  

# Cross-Encoder는 리랭커
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L6-v2")

# 문단 단위로 분리 후, Bi-Encoder로 인코딩
wikipedia_filepath = "simplewiki-2020-11-01.jsonl.gz" 

if not os.path.exists(wikipedia_filepath):
    http_get("http://sbert.net/datasets/simplewiki-2020-11-01.jsonl.gz", wikipedia_filepath)

passages = []
with gzip.open(wikipedia_filepath, "rt", encoding="utf8") as fIn:
    for line in fIn:
        data = json.loads(line.strip())

        # 모든 문단 저장
        # passages.extend(data['paragraphs'])

        # 첫 번째 문단만
        passages.append(data["paragraphs"][0])

print("Passages:", len(passages))

corpus_embeddings = bi_encoder.encode(passages, convert_to_tensor=True, show_progress_bar=True)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 12341.72it/s]


Passages: 169597


Batches: 100%|██████████| 5300/5300 [00:25<00:00, 204.66it/s]


In [6]:
# 키워드 검색과 비교를 위해 BM25 사용
import string
import numpy as np

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction import _stop_words
from tqdm.autonotebook import tqdm

def bm25_tokenizer(text):# 소문자로 정규화 및 불용어 삭제
    tokenized_doc = []
    for token in text.lower().split(): 
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc


tokenized_corpus = []
for passage in tqdm(passages):
    tokenized_corpus.append(bm25_tokenizer(passage))

bm25 = BM25Okapi(tokenized_corpus)

100%|██████████| 169597/169597 [00:01<00:00, 142412.31it/s]


In [ ]:
# 쿼리에 답이 되는 문서들 검색
def search(query):
    print("Query: ", query)
    # 키워드 검색 (BM25)
    bm25_scores = bm25.get_scores(bm25_tokenizer(query))
    top_n = np.argpartition(bm25_scores,-5)[-5:] # top-5
    bm25_hits = [{'corpus_id':idx, "score":bm25_scores[idx]} for idx in top_n] # top-k 문서 저장
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True) # top-k 문서 점수순으로 내림차순

    print('Top-3 Lexical Search (BM25) hits')
    for hit in bm25_hits[:3]:
        print("\t{:.3f}\t{}".format(hit['score'], passages[hit['corpus_id']].replace("\n", " ")))

    # 시맨틱 검색
    query_emb = bi_encoder.encode(query, convert_to_tensor=True)
    query_emb = query_emb.cuda()
    hits = semantic_search(query_emb, corpus_embeddings, top_k=top_k)
    hits = hits[0] #첫 번째 쿼리에 대한 검색 결과, 쿼리가 여러 개 있을 수 있음

    # 리랭크
    # Cross_Encoder를 통해 검색된 모든 문단들에 대해 점수 
    cross_inp = [[query, passages[hit['corpus_id']]] for hit in hits]
    cross_scores = cross_encoder.predict(cross_inp)

    for idx in range(len(cross_scores)):
        hits[idx]['cross-score'] = cross_scores[idx]

    # Bi-Encoder로 부터 top-3 문서 추출
    print('\n--------------------\n')
    print("Bi-Encoder 검색된 Top-3 문서들")
    hits = sorted(hits, key=lambda x: x['score'], reverse=True)
    for hit in hits[:3]:
        print("\t{:.3f}\t{}".format(hit["score"], passages[hit["corpus_id"]].replace("\n", " ")))

    # 리랭커에 의해 조정된 top-3
        print("\n-------------------------\n")
    print("리랭커(Cross Encoder)로 부터 조정된 Top-3 문서")
    hits = sorted(hits, key=lambda x: x["cross-score"], reverse=True)
    for hit in hits[0:3]:
        print("\t{:.3f}\t{}".format(hit["cross-score"], passages[hit["corpus_id"]].replace("\n", " ")))
search(query="What is the capital of the United States?")

Query:  What is the capital of the United States?
Top-3 Lexical Search (BM25) hits
	13.316	Capital punishment (the death penalty) has existed in the United States since before the United States was a country. As of 2017, capital punishment is legal in 30 of the 50 states. The federal government (including the United States military) also uses capital punishment.
	11.434	Ohio is one of the 50 states in the United States. Its capital is Columbus. Columbus also is the largest city in Ohio.
	11.179	Nevada is one of the United States' states. Its capital is Carson City. Other big cities are Las Vegas and Reno.

--------------------

Bi-Encoder 검색된 Top-3 문서들
	0.622	Cities in the United States:

-------------------------

	0.597	The United States Capitol is the building where the United States Congress meets. It is the center of the legislative branch of the U.S. federal government. It is in Washington, D.C., on top of Capitol Hill at the east end of the National Mall.

----------------------

In [8]:
search(query="What is the best orchestra in the world?")

Query:  What is the best orchestra in the world?
Top-3 Lexical Search (BM25) hits
	15.328	The BBC Symphony Orchestra is the main orchestra of the British Broadcasting Corporation. It is one of the best orchestras in Britain.
	15.320	The NHK Symphony Orchestra is a Japanese orchestra based in Tokyo, Japan. In Japanese it is written: NHK交響楽団, pronounced: Enueichikei Kōkyō Gakudan. When the orchestra was started in 1926 it was called "New Symphony Orchestra". It was the first large professional orchestra in Japan. Later, it changed its name to "Japan Symphony Orchestra". In 1951 it started to get money from the Japanese radio station NHK (Nippon Hōsō Kyōkai), so it changed its name again to the name it has now. It is thought of as the best orchestra in Japan. They have played in many parts of the world, including at the BBC Proms in London.
	14.079	The Bamberger Symphoniker (Bamberg Symphony Orchestra) is a world-famous orchestra from the city of Bamberg, Germany. It was formed in 1946. M

In [9]:
search(query="Number countries Europe")

Query:  Number countries Europe
Top-3 Lexical Search (BM25) hits
	13.795	Amy MacDonald is a Scottish singer and songwriter. She became famous in 2007 with her first album "This Is The Life" and her first single "Poison Prince". She has become even more successful in Europe since her single "This Is The Life" charted at number 1 in many European countries.
	13.758	The Croatian language is spoken mainly throughout the countries of Croatia and Bosnia and Herzegovina and in the surrounding countries of Europe.
	13.019	Organization for Security and Co-operation in Europe (OSCE) is an international organization for peace and human rights. Presently, it has 57 countries as its members. Most of the member countries of the OSCE are from Europe, the Caucasus, Central Asia and North America.

--------------------

Bi-Encoder 검색된 Top-3 문서들
	0.538	The Council of Europe (, ) is an international organization of 47 member states in the European region. One of its first successes was the European Conve

In [10]:
search(query="When did the cold war end?")

Query:  When did the cold war end?
Top-3 Lexical Search (BM25) hits
	17.374	The Cold War was the tense relationship between the United States (and its allies), and the Soviet Union (the USSR and its allies) between the end of World War II and the fall of the Soviet Union. It is called the "Cold" War because the US and the USSR never actually fought each other directly. Instead, they opposed each other in conflicts known as proxy wars, where each country chose a side to support.
	17.291	The Reagan Doctrine was a document by the United States under the Reagan Administration. It was about being against the global influence of the Soviet Union during the final years of the Cold War. The doctrine lasted for less than a decade, it was the most important document of United States foreign policy from the early 1980s until the end of the Cold War in 1991.
	15.420	Cold Norton is a village and civil parish in Maldon District, Essex, England. In 2001 there were 1103 people living in Cold Norton. C